# vForge — 05. Full pipeline (Sprint demo)

Single notebook that runs the **whole pipeline** for the Sprint blog:
1. Generate dataset (Together AI)
2. Fine-tune on TPU (LoRA)
3. Benchmark with vLLM
4. Compare to a GPU baseline (loaded from `benchmarks/results/`)

Run on Colab → TPU runtime.

## 1. Install

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'jax'
!pip -q install -U 'jax[tpu]' keras==3.6.0 keras-hub datasets vllm httpx huggingface_hub

## 2. Generate dataset

In [ ]:
# (uses code from 01_dataset_generation.ipynb — abbreviated here)
import os, httpx, json, asyncio
os.environ['TOGETHER_API_KEY'] = 'YOUR_KEY_HERE'
# ... see 01_dataset_generation.ipynb for the full gen() helper ...
# rows = await gen('code', 'Pandas one-liners', n=200)
# Path('data/code.jsonl').write_text('\n'.join(json.dumps(r) for r in rows))

## 3. Fine-tune on TPU

In [ ]:
# (see 02_finetune_tpu.ipynb for the full version)
import keras, keras_hub, time
devs = keras.distribution.list_devices()
if devs and any('tpu' in d.lower() for d in devs):
    keras.distribution.set_distribution(keras.distribution.DataParallel(devices=devs))
causal = keras_hub.models.CausalLM.from_preset('gemma2_2b_en')
causal.backbone.enable_lora(rank=8)
# causal.fit(texts, batch_size=4, epochs=1)
# causal.save_weights('out_tpu/lora.weights.h5')

## 4. Benchmark vLLM (TPU)

In [ ]:
# from vllm import LLM, SamplingParams
# (see 04_benchmark_vllm.ipynb)
# metrics_tpu = {...}
# Path('benchmarks/results/tpu_pipeline.json').write_text(json.dumps({'metrics': metrics_tpu}))

## 5. Compare with GPU baseline

In [ ]:
from pathlib import Path
import json
tpu = json.loads(Path('benchmarks/results/tpu_pipeline.json').read_text())['metrics']
gpu = json.loads(Path('benchmarks/results/gpu_pipeline.json').read_text())['metrics']
print('TPU tok/s:', tpu['throughput_tokens_per_sec'], '  GPU tok/s:', gpu['throughput_tokens_per_sec'])
print('TPU cost/1M:', tpu['cost_per_1m_tokens_usd'], '  GPU cost/1M:', gpu['cost_per_1m_tokens_usd'])

## Done
- Adapter + metrics in `out_tpu/`.
- Benchmark JSONs in `benchmarks/results/`.
- Render charts: `python benchmarks/analysis.py --output benchmarks/charts/`.